# harness_agents

> Harness-aware bandit and circuit members (Phase 7).

Spec: `specs/multiagent/phase7/`.

In [ ]:
#| default_exp harness_agents

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from collections import deque

import numpy as np

from explRL.circuit import goalStep
from explRL.harness import Blackboard, Lead, MessageBus
from explRL.team_agents import PatchMember, banditAgent, CircuitPatchMember


class CommBandit(banditAgent):
    def __init__(self, channel=None, w=1.0, K=None, cUCB=1.0, H=10.0):
        super().__init__(K=K, cUCB=cUCB, H=H)
        self.channel, self.w, self.reports = channel, float(w), None

    def event(self, world, t):
        self._world, self._t = world, t
        if isinstance(self.channel, MessageBus):
            for _, j, v, _ in self.channel.take(self.id):
                if self.w != 0:
                    self._store()[j].append(v)
        return super().event(world, t)

    def _store(self):
        if self.reports is None:
            self.reports = [deque(maxlen=self.K) for _ in range(self._world.L)]
        return self.reports

    def _stats(self, obs):
        if self.channel is None or self.w == 0:
            return super()._stats(obs)
        if self.rates is None:
            self.rates = [deque(maxlen=self.K) for _ in range(self._world.L)]
        S = np.array([sum(r) for r in self.rates], float)
        n = np.array([len(r) for r in self.rates], float)
        rep = self._store()
        S = S + self.w * np.array([sum(r) for r in rep], float)
        n = n + self.w * np.array([len(r) for r in rep], float)
        if isinstance(self.channel, Blackboard):
            Sv, Cv = self.channel.view(self.id, self._t)
            S, n = S + self.w * Sv, n + self.w * Cv
        return np.where(n > 0, S / np.where(n > 0, n, 1), 0.0), n

    def learn(self, obs, target, reward, info):
        super().learn(obs, target, reward, info)
        if info['kind'] != 'stay':
            return
        rate, loc = reward / info['dur'], obs['location']
        if isinstance(self.channel, Blackboard):
            self.channel.write(self.id, loc, rate, self._t)
        elif isinstance(self.channel, MessageBus) and self.channel.due(self.id, self._t):
            own = [(np.mean(r), j) for j, r in enumerate(self.rates) if len(r)]
            if own:
                v, j = max(own, key=lambda x: (x[0], -x[1]))
                self.channel.send(self._world, self.id, j, v, self._t)

    def estimates(self, t):
        self._t = t
        return self._stats(None)


class LeadFollower(PatchMember):
    def __init__(self, lead):
        self.lead = lead

    def event(self, world, t):
        self._t = t
        return super().event(world, t)

    def act(self, obs):
        return self.lead.assign(self.id, obs['location'], obs['travel'], self._t)

    def learn(self, obs, target, reward, info):
        if info['kind'] == 'stay':
            self.lead.report(self.id, obs['location'], reward / info['dur'], info['nHere'], self._t)


class CircuitCommMember(CircuitPatchMember):
    def __init__(self, agent, channel=None, w=1):
        if isinstance(channel, Lead):
            raise TypeError('circuit members cannot follow a lead')
        super().__init__(agent)
        self.channel, self.w, self.own = channel, int(w), {}

    def reward(self, world, t, fix):
        R = super().reward(world, t, fix)
        self._net, self._fix = self._lastNet, fix
        return R

    def event(self, world, t):
        out = super().event(world, t)
        if self.channel is not None:
            self._share(world, t)
            if self.w > 0:
                self._read(t)
            elif isinstance(self.channel, MessageBus):
                self.channel.take(self.id)
        return out

    def _own_rate(self, j):
        s, c = self.own[j]
        return s / c if c else 0.0

    def _share(self, world, t):
        ag = self.agent
        a, rate = ag.a, (self._net / self._fix if self._fix else 0.0)
        if a not in self.own:
            self.own[a] = [0.0, 0]
        self.own[a][0] += self._net
        self.own[a][1] += 1
        if isinstance(self.channel, Blackboard):
            self.channel.write(self.id, a, rate, t)
        elif isinstance(self.channel, MessageBus) and self.channel.due(self.id, t) and self.own:
            j = max(self.own, key=lambda k: (self._own_rate(k), -k))
            self.channel.send(world, self.id, j, self._own_rate(j), t)

    def _read(self, t):
        ag, p = self.agent, self.agent.params
        reports = []
        if isinstance(self.channel, MessageBus):
            reports = [(j, v) for _, j, v, _ in self.channel.take(self.id)]
        elif isinstance(self.channel, Blackboard):
            S, C = self.channel.view(self.id, t)
            if (C > 0).any():
                j = int(np.argmax(np.where(C > 0, S / C, -np.inf)))
                reports = [(j, float(S[j] / C[j]))]
        for j, value in reports:
            if value != 0 and ag.rl.learnG:
                cRep = (ag.c + j - ag.a) % p.nCirc
                for _ in range(self.w):
                    ag.policy.wG = goalStep(ag.policy.wG, cRep, np.sign(value), p)

## Tests

In [ ]:
import numpy as np
from collections import deque
from fastcore.test import test_eq, test_close
from explRL.harness import Blackboard, MessageBus, Lead
from explRL.harness_agents import CommBandit, LeadFollower, CircuitCommMember
from explRL.harness_runs import benchWorld
from explRL.core import loadCircuitAgent
from explRL.members import CircuitMember
from explRL.multi import runTeam, spawnRngs
from explRL.patchworld import TeamPatchWorld
from explRL.team_agents import banditAgent, CircuitPatchMember
from explRL.circuit import goalStep

def same_ledger(a, b):
    test_eq(len(a), len(b))
    for ra, rb in zip(a, b):
        test_eq(len(ra), len(rb))
        for x, y in zip(ra, rb):
            if isinstance(x, float) and np.isnan(x):
                assert np.isnan(y)
            else:
                test_eq(x, y)

def same_hist(h0, h1):
    test_eq(len(h0.records), len(h1.records))
    for a, b in zip(h0.records, h1.records):
        test_eq((a.t, a.dt, a.location, a.reward), (b.t, b.dt, b.location, b.reward))

for seed in (0, 1, 2):
    for mk in ('none', 'bb', 'msg'):
        w0, w1 = benchWorld(), benchWorld()
        w0.reset(seed, 8); w1.reset(seed, 8)
        ch = None if mk == 'none' else Blackboard(w0.L, n=8) if mk == 'bb' else MessageBus(8, rngs=spawnRngs(seed + 2000, 8))
        m0 = [banditAgent() for _ in range(8)]
        m1 = [CommBandit(ch, w=0.0) for _ in range(8)]
        h0 = runTeam(m0, w0, 300.0)
        h1 = runTeam(m1, w1, 300.0)
        same_ledger(w0.ledger, w1.ledger)
        same_hist(h0, h1)
        if ch is not None:   # w = 0 members still share (FR-7), so the bridge also shows sharing changes nothing
            assert sum(s['written'] + s['sent'] for s in ch.stats.values()) > 0

# AC-4: messages on (w = 1) leave the world's draws alone: the same patch sites as with no channel
sites = []
for ch in (None, MessageBus(8, rngs=spawnRngs(2000, 8))):
    wm = benchWorld(); wm.reset(0, 8)
    runTeam([CommBandit(ch) for _ in range(8)], wm, 300.0)
    sites.append(list(wm.site))
test_eq(sites[0], sites[1])

# AC-4: message carries sender's own best stay mean
w = benchWorld(); w.reset(0, 2)
bus = MessageBus(2, k=1, period=5.0, radius=np.inf, rngs=spawnRngs(11, 2))
cb = CommBandit(bus, w=1.0)
cb.id = 0
cb.rates = [deque([3.0]), deque([1.0])]
cb._world, cb._t = w, 0.0
bus.window[0] = -1
cb.learn({'location': 0}, 0, 3.0, {'kind': 'stay', 'dur': 1.0})
got = bus.take(1)
test_eq(len(got), 1)
test_close(got[0][2], 3.0)

# AC-6
bb6 = Blackboard(2, n=2, lam=0.0)
cb6 = CommBandit(bb6, w=0.5, K=3)
cb6.rates = [deque([2.0]), deque()]
cb6.reports = [deque(), deque([4.0])]
cb6._world, cb6._t, cb6.id = benchWorld(), 0.0, 0
bb6.write(1, 1, 6.0, 0.0)
mean, n = cb6._stats({'travel': np.ones(2)})
test_close(mean[1], 5.0)
test_close(n[1], 1.0)
test_eq(cb6.act({'travel': np.array([1., 1.]), 'location': 0}), 1)
# a location known only from reports counts as visited (else the nearest unvisited, 1, would be chosen)
cbV = CommBandit(MessageBus(1, rngs=spawnRngs(0, 1)), w=1.0); cbV.id, cbV._world = 0, benchWorld()
cbV.rates = [deque([2.0]), deque(), deque()]
cbV.reports = [deque(), deque([1.0]), deque()]
test_eq(cbV.act({'travel': np.array([0., 1., 5.]), 'location': 0}), 2)
# K limits own stays and stored reports separately
cbK = CommBandit(MessageBus(2, rngs=spawnRngs(0, 2)), w=1.0, K=2); cbK.id, cbK._world = 0, benchWorld()
cbK.rates = [deque(maxlen=2) for _ in range(cbK._world.L)]
for v in (1.0, 2.0, 3.0):
    cbK.rates[0].append(v); cbK._store()[0].append(10 * v)
mK, nK = cbK._stats(None)
test_eq(float(nK[0]), 4.0); test_close(mK[0], (2 + 3 + 20 + 30) / 4)

w = benchWorld(); w.reset(1, 2)
lead = Lead(w.L, 2, lam=0.0)
followers = [LeadFollower(lead) for _ in range(2)]
hL = runTeam(followers, w, 50.0)
# followers go where the lead assigns and report every stay
assert all(r.location == r.extras[0] for r in hL.records)
for i in range(2):
    test_eq(lead.assigned[i], hL.forAgent(i)[-1].extras[0])
test_eq(sum(lead.stats[i]['written'] for i in range(2)), sum(r[3] == 'stay' for r in w.ledger))

CIRC_N = len(CircuitMember.extraNames)
for symmetric in (True, False):
    for seed in (0, 1, 2):
        world = TeamPatchWorld.zoneConfig(symmetric=symmetric)
        world.reset(seed, 1)
        h0 = runTeam([CircuitPatchMember(loadCircuitAgent(seed=seed))], world, 300.0)
        world.reset(seed, 1)
        h1 = runTeam([CircuitCommMember(loadCircuitAgent(seed=seed))], world, 300.0)
        same_hist(h0, h1)
        world.reset(seed, 1)
        h2 = runTeam([CircuitCommMember(loadCircuitAgent(seed=seed), Blackboard(world.L, n=1), w=0)], world, 300.0)
        same_hist(h0, h2)

# AC-7
ag = loadCircuitAgent(seed=0)
bb7 = Blackboard(32, n=2, lam=0.0)
mem = CircuitCommMember(ag, bb7, w=2); mem.id = 0
p = ag.params
j = 3
bb7.write(1, j, 1.0, 0.0)
wG0 = ag.policy.wG.copy()
cRep = (ag.c + j - ag.a) % p.nCirc
wGexp = wG0.copy()
for _ in range(2):
    wGexp = goalStep(wGexp, cRep, 1, p)
mem._read(0.0)
test_close(ag.policy.wG, wGexp, eps=0.01)
bb7.write(1, j, -1.0, 1.0)
ag.policy.wG = wG0.copy()
wGexp = wG0.copy()
for _ in range(2):
    wGexp = goalStep(wGexp, cRep, -1, p)
mem._read(1.0)
test_close(ag.policy.wG, wGexp, eps=0.01)
ag.rl.learnG = False
ag.policy.wG = wG0.copy()
mem._read(1.0)
test_close(ag.policy.wG, wG0)

try:
    CircuitCommMember(loadCircuitAgent(seed=0), Lead(32, 1))
except TypeError:
    pass
else:
    raise AssertionError('expected TypeError')


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()